In [1]:
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

today = datetime.now()
date_str = today.strftime("%Y%m%d")
filename = f"請求書_{date_str}.xlsx"

wb = Workbook()
ws = wb.active
ws.title = "請求書"

# 列幅（A〜G）
widths = {"A": 2.5, "B": 18, "C": 10, "D": 12, "E": 12, "F": 8, "G": 14}
for col, w in widths.items():
    ws.column_dimensions[col].width = w

# スタイル
bold = Font(bold=True)
title_font = Font(size=16, bold=True)
thin = Side(style="thin", color="000000")
border_thin = Border(left=thin, right=thin, top=thin, bottom=thin)
center = Alignment(horizontal="center", vertical="center")
left = Alignment(horizontal="left", vertical="center")
right = Alignment(horizontal="right", vertical="center")

# タイトル
ws["B2"] = "請求書"
ws["B2"].font = title_font

# 固定データ（会社情報）
ws["B4"] = "株式会社ABC"
ws["B4"].font = bold
ws["B5"] = "〒101-0022 東京都千代田区神田練塀町300"
ws["B6"] = "TEL:03-1234-5678 FAX:03-1234-5678"
ws["B7"] = "担当者名:鈴木一郎 様"
ws["B7"].font = bold

# No. / 日付
ws["F4"] = "No."
ws["F4"].font = bold
ws["G4"] = "0001"
ws["G4"].font = bold

ws["F5"] = "日付"
ws["F5"].font = bold
ws["G5"] = today
ws["G5"].number_format = "yyyy/mm/dd"  # 2023/06/06形式（YYYY/MM/DD）

# 明細ヘッダ
headers = ["商品名", "数量", "単価", "金額"]
for i, h in enumerate(headers, start=2):  # B=2
    c = ws.cell(row=10, column=i, value=h)
    c.font = bold
    c.alignment = center
    c.fill = PatternFill("solid", fgColor="F2F2F2")
    c.border = border_thin

# 明細（例）
items = [
    ("商品A", 2, 10000),
    ("商品B", 1, 15000),
]
start_row = 11
for r, (name, qty, unit) in enumerate(items, start=start_row):
    ws[f"B{r}"] = name
    ws[f"C{r}"] = qty
    ws[f"D{r}"] = unit
    ws[f"E{r}"] = f"=C{r}*D{r}"

    ws[f"B{r}"].alignment = left
    ws[f"C{r}"].alignment = center
    ws[f"D{r}"].alignment = right
    ws[f"E{r}"].alignment = right

    for col in ["B", "C", "D", "E"]:
        ws[f"{col}{r}"].border = border_thin

# 小計行（画像の 35000 の行）
sum_row = start_row + len(items)
ws[f"E{sum_row}"] = f"=SUM(E{start_row}:E{sum_row-1})"
ws[f"E{sum_row}"].alignment = right
for col in ["B", "C", "D", "E"]:
    ws[f"{col}{sum_row}"].border = border_thin

# 合計欄（合計・消費税・税込合計）
ws["B15"] = "合計"
ws["B16"] = "消費税"
ws["B17"] = "税込合計"
for r in [15, 16, 17]:
    ws[f"B{r}"].font = bold
    ws[f"B{r}"].alignment = left

ws["E15"] = f"=E{sum_row}"
ws["E16"] = "=ROUND(E15*0.10,0)"
ws["E17"] = "=E15+E16"
for r in [15, 16, 17]:
    ws[f"E{r}"].alignment = right

# 数値表示（カンマなし）
for cell in ["D11", "E11", "D12", "E12", f"E{sum_row}", "E15", "E16", "E17"]:
    ws[cell].number_format = "0"

# No/日付の整列
for addr in ["F4", "F5"]:
    ws[addr].alignment = left
for addr in ["G4", "G5"]:
    ws[addr].alignment = right

# 行高（見た目調整）
ws.row_dimensions[2].height = 24
for r in range(10, 18):
    ws.row_dimensions[r].height = 18

wb.save(filename)
print(f"作成しました: {filename}")

作成しました: 請求書_20260221.xlsx
